In [3]:
import pandas as pd
import datetime
import warnings
import time
import numpy as np
import sys
from gf_ck_gen import save_google_flights_landing
from gf_req import requests_get_with_playwright_cookies
from gf_parser import parse_google_flights_html_text

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 30)
warnings.filterwarnings("ignore")


# Cookies Generation

Esegui questa cella quando vuoi aggiornare/salvare nuovi cookies. Le celle successive useranno automaticamente gli ultimi cookies salvati in `gf_landing_artifacts`, senza passare `result["cookies_path"]`.

In [6]:
import subprocess
import os

# Esegui lo script standalone per salvare i cookies
notebook_dir = r"c:\Users\valerio.vescio\Desktop\Wescio\Monitor Voli\Codici Originali\gf_req_v2 2\gf_req_v2"
result = subprocess.run(
    [sys.executable, os.path.join(notebook_dir, "save_cookies_standalone.py")],
    capture_output=True,
    text=True,
    cwd=notebook_dir
)
print(result.stdout)
if result.stderr:
    print("Errori:", result.stderr)


goto: https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20to%20PMO%20from%20FCO%20on%202026-08-09%20oneway%20economy%20nonstops
cookie banner clicked: True
final url before save: https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights+to+PMO+from+FCO+on+2026-08-09+oneway+economy+nonstops
{
  "input_url": "https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20to%20PMO%20from%20FCO%20on%202026-08-09%20oneway%20economy%20nonstops",
  "final_url": "https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights+to+PMO+from+FCO+on+2026-08-09+oneway+economy+nonstops",
  "cookie_banner_clicked": true,
  "cookies_path": "gf_landing_artifacts\\20260619_152923_cookies.json",
  "cookies_count": 3,
  "html_path": "gf_landing_artifacts\\20260619_152923_landing.html",
  "html_bytes": 2714230
}

âœ… Cookies salvati con successo!
{'input_url': 'https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20to%20PMO%20from%20FCO%20on%202026-08-09%20oneway%20economy%20nonstops', 'final_

# Request Generation

V2: `cookies_path` è opzionale. Se non viene passato, `gf_req` usa l’ultimo file `*_cookies.json` trovato in `gf_landing_artifacts`. L’output `out` contiene direttamente anche `out["html"]`, cioè `response.text`.

In [9]:

url = "https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20from%20MIL%20to%20PMO%20on%202026-06-21%20oneway%20economy%20nonstops"
out = requests_get_with_playwright_cookies(url,out_html_path="gf_landing_artifacts/requests_get_response.html")

info={k: v for k, v in out.items() if k not in {"html", "response_text"}}
info

{'status_code': 200,
 'ok': True,
 'final_url': 'https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20from%20MIL%20to%20PMO%20on%202026-06-21%20oneway%20economy%20nonstops',
 'cookies_path': 'gf_landing_artifacts\\20260619_152923_cookies.json',
 'cookies_source': 'latest_saved',
 'saved_path': 'gf_landing_artifacts\\requests_get_response.html',
 'response_bytes': 2416902,
 'html_bytes': 2416902,
 'content_type': 'text/html; charset=utf-8',
 'encoding': 'utf-8'}

# Data Parser

V2: il parser riceve direttamente l’HTML in memoria da `out["html"]`, senza leggere il file locale.

In [10]:
df, info = parse_google_flights_html_text(
    out["html"],
    include_raw_label=False)

print("📊 Parser Results:")
print(f"Flights found: {len(df)}")
print(f"Info: {info}")
print()
df.head()


📊 Parser Results:
Flights found: 15
Info: {'html_path': None, 'html_bytes': 2416902, 'cards_li_pIav2d_found': 30, 'flight_card_candidates_found': 30, 'raw_rows_before_dedupe': 30, 'dedupe': True, 'rows_parsed': 15, 'empty': False, 'warnings': [], 'title': 'Da Milano a Palermo | Google Voli', 'source_url': 'https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights from MIL to PMO on 2026-06-21 oneway economy nonstops&hl=it', 'query_text': 'Flights from MIL to PMO on 2026-06-21 oneway economy nonstops', 'query_origin': None, 'query_destination': None, 'query_depart_date': None}



,result_index,flight_id,flight_id_source,section,price_eur,currency,airline,operated_by,carrier_code,flight_number,flight_departure_date,origin,destination,depart_time,arrive_time,...,stops_count,origin_airport_name,destination_airport_name,depart_day_text,arrive_day_text,emissions_kg_co2e,baggage_cabin_not_included,flight_segments_count,flight_segment_ids,tim_itinerary,travelimpactmodel_url,card_id,airport_codes_found,query_text,flight_segments_json
0,0,U23511,travelimpactmodel_itinerary,Voli più pertinenti,111,EUR,easyJet,NaN,U2,3511,2026-06-21,MXP,PMO,21:55,23:40,...,0,Aeroporto di Milano Malpensa,Aeroporto di Palermo Falcone e Borsellino,"domenica, giugno 21","domenica, giugno 21",87.0,True,1,U23511,MXP-PMO-U2-3511-20260621,https://www.travelimpactmodel.org/lookup/fligh...,rdXEnb,"MXP,PMO",Flights from MIL to PMO on 2026-06-21 oneway e...,"[{""origin"": ""MXP"", ""destination"": ""PMO"", ""carr..."
1,3,FR6458,travelimpactmodel_itinerary,Altri voli,144,EUR,Ryanair,Malta Air,FR,6458,2026-06-21,BGY,PMO,08:10,09:50,...,0,Milan Bergamo Airport,Aeroporto di Palermo Falcone e Borsellino,"domenica, giugno 21","domenica, giugno 21",84.0,True,1,FR6458,BGY-PMO-FR-6458-20260621,https://www.travelimpactmodel.org/lookup/fligh...,JHmdBb,"BGY,PMO",Flights from MIL to PMO on 2026-06-21 oneway e...,"[{""origin"": ""BGY"", ""destination"": ""PMO"", ""carr..."
2,4,U23501,travelimpactmodel_itinerary,Altri voli,151,EUR,easyJet,NaN,U2,3501,2026-06-21,MXP,PMO,08:00,09:45,...,0,Aeroporto di Milano Malpensa,Aeroporto di Palermo Falcone e Borsellino,"domenica, giugno 21","domenica, giugno 21",88.0,True,1,U23501,MXP-PMO-U2-3501-20260621,https://www.travelimpactmodel.org/lookup/fligh...,WHGkzf,"MXP,PMO",Flights from MIL to PMO on 2026-06-21 oneway e...,"[{""origin"": ""MXP"", ""destination"": ""PMO"", ""carr..."
3,1,AZ1773,travelimpactmodel_itinerary,Voli più pertinenti,168,EUR,ITA,NaN,AZ,1773,2026-06-21,LIN,PMO,08:25,10:05,...,0,Aeroporto di Milano Linate,Aeroporto di Palermo Falcone e Borsellino,"domenica, giugno 21","domenica, giugno 21",99.0,False,1,AZ1773,LIN-PMO-AZ-1773-20260621,https://www.travelimpactmodel.org/lookup/fligh...,EbXNlb,"LIN,PMO",Flights from MIL to PMO on 2026-06-21 oneway e...,"[{""origin"": ""LIN"", ""destination"": ""PMO"", ""carr..."
4,2,FR6149,travelimpactmodel_itinerary,Voli più pertinenti,169,EUR,Ryanair,Malta Air,FR,6149,2026-06-21,BGY,PMO,17:30,19:10,...,0,Milan Bergamo Airport,Aeroporto di Palermo Falcone e Borsellino,"domenica, giugno 21","domenica, giugno 21",72.0,True,1,FR6149,BGY-PMO-FR-6149-20260621,https://www.travelimpactmodel.org/lookup/fligh...,c80Vc,"BGY,PMO",Flights from MIL to PMO on 2026-06-21 oneway e...,"[{""origin"": ""BGY"", ""destination"": ""PMO"", ""carr..."


# Plan Gen

In [4]:
def url_compose(city_dep,city_arr,target_date):
    return f"https://www.google.com/travel/flights?gl=IT&hl=it&q=Flights%20from%20{city_dep}%20to%20{city_arr}%20on%20{target_date}%20oneway%20economy%20nonstops"

def plan_gen(directory):
    plan_df=pd.read_excel(directory+"plan_sum26.xlsx")
    sigle_df=pd.read_excel(directory+"plan_sum26.xlsx",sheet_name='sigle')
    plan_df=pd.merge(plan_df,sigle_df.rename(columns={"city":"origin","sigla":"city_dep"}),on='origin')
    plan_df=pd.merge(plan_df,sigle_df.rename(columns={"city":"destination","sigla":"city_arr"}),on='destination')
    plan_df["url"]=plan_df.apply(lambda x:url_compose(x["city_dep"],x["city_arr"],x["target_date"]),axis=1)

    return plan_df

directory="plans/"
plan_df=plan_gen(directory)
plan_df.head()

,origin,destination,target_date,cluster,flag,city_dep,city_arr,url
0,Alghero,Milano,2026-08-09,from,sum26_ext,AHO,MIL,https://www.google.com/travel/flights?gl=IT&hl...
1,Alghero,Milano,2026-08-16,from,sum26_ext,AHO,MIL,https://www.google.com/travel/flights?gl=IT&hl...
2,Alghero,Milano,2026-08-23,from,sum26_ext,AHO,MIL,https://www.google.com/travel/flights?gl=IT&hl...
3,Alghero,Milano,2026-08-30,from,sum26_ext,AHO,MIL,https://www.google.com/travel/flights?gl=IT&hl...
4,Alghero,Roma,2026-08-09,from,sum26_ext,AHO,ROM,https://www.google.com/travel/flights?gl=IT&hl...


# Monitor Gen

In [5]:
def monitor_gen(plan_df):
    update_date=datetime.datetime.now().date()
    start_time=time.time()
    i=0
    for row in plan_df.to_dict(orient='records'):
        print("-------------------------","N.:",i,"origin:",row["origin"],"destination:",row["destination"],"target_date:",row["target_date"])
        url=row["url"]
        out = requests_get_with_playwright_cookies(url,out_html_path="gf_landing_artifacts/requests_get_response.html")
        loc_df,info=parse_google_flights_html_text(out["html"],include_raw_label=False,)
        print("N. results:",len(loc_df),"time:",np.round(time.time()-start_time,2))
        if len(loc_df)==0:
            print("$$$$$$$$$$$$ Attention, no Data $$$$$$$$$$$$")
        loc_df.to_excel(f"output/{str(update_date)}_{row['city_dep']}_{row['city_arr']}_{row['target_date']}.xlsx")
        i+=1

        
monitor_gen(plan_df)


------------------------- N.: 0 origin: Alghero destination: Milano target_date: 2026-08-09
N. results: 5 time: 1.35
------------------------- N.: 1 origin: Alghero destination: Milano target_date: 2026-08-16
N. results: 5 time: 3.17
------------------------- N.: 2 origin: Alghero destination: Milano target_date: 2026-08-23
N. results: 5 time: 4.84
------------------------- N.: 3 origin: Alghero destination: Milano target_date: 2026-08-30
N. results: 5 time: 5.92
------------------------- N.: 4 origin: Alghero destination: Roma target_date: 2026-08-09
N. results: 5 time: 7.74
------------------------- N.: 5 origin: Alghero destination: Roma target_date: 2026-08-16
N. results: 5 time: 9.14
------------------------- N.: 6 origin: Alghero destination: Roma target_date: 2026-08-23
N. results: 5 time: 10.08
------------------------- N.: 7 origin: Alghero destination: Roma target_date: 2026-08-30
N. results: 5 time: 11.53
------------------------- N.: 8 origin: Bari destination: Milano targe

KeyboardInterrupt: 